# FloodA5 notebook 6 — eleven meshes, one solver

FloodA5's solver works on any mesh of convex polygons: it sees a mesh only
through each cell's area and centre and each face's length, normal and
neighbours (`docs/MESHES.md` §1, "the mesh contract"). So the mesh can be
chosen like any other option, and its effect measured. This notebook does
that for eleven meshes at the same mean cell size, about 22 m: the square
grid FloodA5 has always had, four other planar meshes, A5 and five other
discrete global grids (DGGS). It shows

1. what each mesh is and looks like, and its geometry as the solver sees it
   (§1–2);
2. how the model behaves on it in four benchmarks — a point source on a flat
   bed (isotropy), a point source on a slope (directional bias), a straight
   channel one cell wide at four orientations, and the Carlisle flood
   against 5 m references (§3–7);
3. **the SGS checkerboard** (§8): until 0.9.0 the sub-grid model (SGS)
   produced a cell-scale checkerboard in the water level on most meshes
   other than A5. §8 measures it per mesh, and is meant to be re-run after
   work on the SGS hydraulics, as a check (set `RERUN_SGS = true` in §0);
4. what each mesh costs (§9), and its advantages and disadvantages (§10).

**In short:** with the local-inertial model the mesh family matters less
than the cell size on a real flood: every mesh comes within 0.92–0.97 of the
5 m full shallow-water storage at Carlisle, within 0.25–0.30 m in level.
The differences show in idealised flows, where each mesh favours the
directions of its faces: square and rotated grids and rHEALPix depend most
on the orientation of a narrow channel, A5 and HEALPix least; HEALPix,
skewed at British latitudes, deflects flow on a slope by 3°. SGS is smooth
on every mesh since 0.9.0 (§8; until then only on A5, ISEA3H and the
jittered Voronoi mesh) and stores 0.70–0.79 of the benchmark on all of them.

**How to run it.** Open it with the kernel **"Julia FloodA5 1.12"** (set-up:
`notebooks/README.md`) and run all cells. The first run makes four cases in
`test/case_runs/` (not tracked by git): the point spread (~5 min), the
planar slope (~10 min), the channel (~45 min) on 3 × 4 threads, and Carlisle
(~10 h; it needs the `carlisle_peak` and `carlisle_peak_fine` runs of
notebook 3, the latter on a GPU). Later runs reuse them and take a few
minutes.

**Words used.** *LI*: local-inertial, `--flow-model standard`, the default.
*SGS*: the sub-grid model, `--flow-model sgs` (each cell keeps the 5 m
terrain as tables). *SWE*: the full shallow-water equations, `--flow-model
swe`. *Cell size*: √(mean cell area). *Valence*: the number of neighbours
of a cell. *Non-orthogonality*: the angle between a face's normal and the
line joining the two cell centres (0° on a square grid).

## 0. Set-up

The cell below finds the repository from the Julia project the kernel was
started with and loads the modules the notebooks use: `FA`
(`analysis/FloodA5Analysis.jl`, reading outputs), `FC` (`cases/FloodA5Cases.jl`,
the test-case registry; these cases are in `cases/registry/mesh_families.jl`)
and `FP` (`visualisation/FloodA5Plots.jl`, the figure style).

- `RUN_CASES = false` only analyses runs that already exist.
- `RERUN_SGS = true` runs the eleven Carlisle SGS simulations again, even
  though they exist: the case runner reuses a run whose command has not
  changed, so after a change to the SGS code this is how §8 sees it. About
  3 hours on 3 × 4 threads. (If the change also affects how the SGS tables
  are built, delete the `*_sgs.parquet` meshes in
  `test/case_runs/mesh_families_carlisle/` and `test/case_runs/carlisle/`
  first, so they are rebuilt.)
- `PARALLEL` runs that many simulations at once, each on `THREADS` threads.

In [ ]:
# The kernel starts Julia with --project=<repository> (or <repository>/ci):
# walk up from that project to the directory holding FloodModel.jl.
const REPO = let d = dirname(something(Base.active_project(), joinpath(pwd(), "Project.toml")))
    while !isfile(joinpath(d, "FloodModel.jl"))
        dirname(d) == d && error("no FloodA5 repository above $(Base.active_project())")
        d = dirname(d)
    end
    d
end
ENV["FLOODA5_NO_VIS"] = "1"            # the runs need no viewer

include(joinpath(REPO, "analysis", "FloodA5Analysis.jl"));     const FA = FloodA5Analysis
include(joinpath(REPO, "cases", "FloodA5Cases.jl"));           const FC = FloodA5Cases
include(joinpath(REPO, "visualisation", "FloodA5Plots.jl"));   const FP = FloodA5Plots
using CairoMakie, Printf, Statistics, TOML
FP.activate!()                         # the FloodA5 figure style

const RUN_CASES = true                 # false: only analyse existing runs
const RERUN_SGS = false                # true: run the Carlisle SGS simulations again (§8)
const PARALLEL  = 3                    # simulations at once
const THREADS   = 4                    # threads per simulation

println("repository : ", REPO)
println("version    : ", strip(read(joinpath(REPO, "VERSION"), String)))
println("case runs  : ", FC.case_root())

Helpers: `mdtable` shows a table (as HTML), `f` formats a number,
`metrics` reads a case's metrics (from its `metrics.toml` if that is newer
than every run, else by analysing the runs), and `MESHES` lists the meshes
with the names used in the tables, the planar ones first.

In [ ]:
"An HTML table from a header and rows of already formatted values."
function mdtable(header, rows)
    esc(x) = replace(string(x), "&" => "&amp;", "<" => "&lt;", ">" => "&gt;")
    cell(tag, x) = "<$tag style='text-align:right'>" * esc(x) * "</$tag>"
    s = "<table><thead><tr>" * join(cell("th", h) for h in header) * "</tr></thead><tbody>"
    for r in rows
        s *= "<tr>" * join(cell("td", v) for v in r) * "</tr>"
    end
    return HTML(s * "</tbody></table>")
end
"`x` to `d` decimals, or as 1.2e-05 if it would round to zero."
f(x, d = 3) = !isfinite(x) ? string(x) :
              (x != 0 && abs(x) < 0.5 * 10.0^-d) ? @sprintf("%.1e", x) : string(round(x; digits = d))
"`x` to `d` decimals, plainly (0.0 for anything that rounds to zero)."
fr(x, d = 2) = !isfinite(x) ? string(x) : string(round(x; digits = d) + 0.0)

"A case's metrics: its metrics.toml if newer than all its runs, else analysed now."
function metrics(name)
    c = FC.case(name)
    p = joinpath(FC.case_dir(c), "metrics.toml")
    outs = [FC.job_output(c, j) for j in c.jobs]
    all(isfile, outs) || error("$name: not every run exists — set RUN_CASES = true")
    isfile(p) && all(o -> mtime(o) < mtime(p), outs) && return TOML.parsefile(p)
    return FC.analyse(c; io = devnull)
end

const MESHES = FC.MESH_FAMILIES        # cart, rotcart, hex, voronoi, voronoi_jit, a5, h3, …
const PLANAR = ["cart", "rotcart", "hex", "voronoi", "voronoi_jit"]
const NAME = Dict("cart" => "Cartesian", "rotcart" => "rotated Cartesian", "hex" => "hexagons",
                  "voronoi" => "centroidal Voronoi", "voronoi_jit" => "jittered Voronoi",
                  "a5" => "A5", "h3" => "H3", "igeo7" => "IGEO7", "isea3h" => "ISEA3H",
                  "healpix" => "HEALPix", "rhealpix" => "rHEALPix")
"Line style of a mesh in the figures: planar meshes dashed, global grids solid."
function mstyle(m)
    grp = m in PLANAR ? PLANAR : filter(x -> !(x in PLANAR), MESHES)
    i = findfirst(==(m), grp)
    return (color = FP.SERIES[i], linestyle = m in PLANAR ? :dash : :solid,
            marker = m in PLANAR ? :rect : :circle)
end
println(join((NAME[m] for m in MESHES), ", "))

## 1. The meshes

### 1.1 What a mesh must provide

A FloodA5 mesh is a GeoParquet file of cells, each with an ID, a polygon
(lon/lat), a centre, and the IDs of the cells that share a side with it
(`docs/MESHES.md` §1). Everything the solver needs — areas, face lengths and
normals, distances between centres — it computes from that, on a sphere. A
file also records its family and, for a DGGS, its library, level and
refinement ratio; since 0.8.1 FloodA5 keeps them through re-saves and copies
them into each output (`FO.mesh_info`).

Meshes come from four generators, all on the command line
(`docs/USER_GUIDE.md` §4.1):

| Generator | Meshes |
|---|---|
| `--meshgen AOI --meshres N` | A5 (pentagons) |
| `--meshgen-cartesian AOI --dx M` | Cartesian: a lon/lat grid, square at the AOI's mid-latitude |
| `mesh/mesh_families.py` | planar meshes: hexagons, rotated Cartesian, Voronoi, or an imported mesh |
| `--meshgen-dggs AOI --dggs FAMILY --dx M` | H3, IGEO7, ISEA3H, HEALPix, rHEALPix |

### 1.2 Eleven meshes at about 22 m

The cases use these meshes, at the cell size of A5 resolution 18 (22.2 m).
The planar ones can be made at any size; a global grid has fixed levels,
each 3 to 9 times smaller in area than the one above, so H3 and rHEALPix
come out about 25% finer and HEALPix 12% coarser. The table gives each
mesh's family properties and its actual cell size at Carlisle.

*Children tile the parent*: the cells of the next level exactly cover each
cell — what lets a model refine or coarsen locally without changing the
coastline of a cell. For A5 this was checked with pya5 (the four children of
a res-17 cell at Carlisle cover the same area as their parent but differ
from it by 64% of it in shape).

In [ ]:
const FAMILY = Dict(
    #  cell shape,  interior valence, equal-area,  refinement (area), children tile the parent
    "cart"        => ("square",        4,  "no (lon/lat)", "any",  "—"),
    "rotcart"     => ("square",        4,  "locally",      "any",  "—"),
    "hex"         => ("hexagon",       6,  "locally",      "any",  "—"),
    "voronoi"     => ("polygon",       "mixed", "locally", "any",  "—"),
    "voronoi_jit" => ("polygon",       "mixed", "locally", "any",  "—"),
    "a5"          => ("pentagon",      5,  "yes",          "×4",   "no"),
    "h3"          => ("hexagon",       6,  "no",           "×7",   "no"),
    "igeo7"       => ("hexagon",       6,  "yes",          "×7",   "no"),
    "isea3h"      => ("hexagon",       6,  "yes",          "×3",   "no"),
    "healpix"     => ("quadrilateral", 4,  "yes",          "×4",   "yes"),
    "rhealpix"    => ("quadrilateral", 4,  "yes",          "×9",   "yes"))
const GEN = Dict("cart" => "--meshgen-cartesian --dx 22.2", "rotcart" => "mesh_families.py rotcart --angle 30",
                 "hex" => "mesh_families.py hex", "voronoi" => "mesh_families.py voronoi --lloyd 5",
                 "voronoi_jit" => "mesh_families.py voronoi --lloyd 0", "a5" => "--meshgen (res 18)")

# Carlisle runs: A5 from carlisle_peak, the others from mesh_families_carlisle
const MFC = FC.case("mesh_families_carlisle")
const PEAK = FC.case("carlisle_peak")
carl_path(m, s) = m == "a5" ? FC.job_output(PEAK, only(filter(j -> j.name == "a5_$s", PEAK.jobs))) :
                              FC.job_output(MFC, only(filter(j -> j.name == "$(m)_$s", MFC.jobs)))
info = Dict(m => (isfile(carl_path(m, "li")) ? FA.FO.mesh_info(carl_path(m, "li")) : Dict{String,Any}()) for m in MESHES)
rows = map(MESHES) do m
    p = FAMILY[m]; mi = info[m]
    lev = haskey(mi, "dggs_level") ? string(mi["dggs_level"]) : m == "a5" ? "18" : "—"
    gen = get(GEN, m, "--meshgen-dggs --dggs $m --dx 22.2")
    (NAME[m], p[1], p[2], p[3], p[4], p[5], lev, gen)
end
mdtable(["Mesh", "Cell", "Valence", "Equal-area", "Refinement", "Children tile parent", "Level", "Generator"], rows)

### 1.3 What they look like

The same 300 × 300 m of Carlisle (around the city centre, where §8 looks for
a checkerboard) on each mesh. The Cartesian grid is a lon/lat grid, so its
cells are squares on the map; the rotated grid is turned by 30°. The global
grids, all made on a sphere (or the WGS84 authalic sphere), are distorted by
their projections at 55° N: HEALPix, in its polar region here, has rhombi
whose diagonals are not at right angles; the hexagonal grids' hexagons are
irregular, and A5's pentagons come in many orientations.

In [ ]:
const LON0, LAT0 = -2.912, 54.898        # Carlisle city centre (Eden and Caldew)
fig = Figure(size = (1100, 830))
for (n, m) in enumerate(MESHES)
    p = carl_path(m, "li"); isfile(p) || continue
    r = FA.load_run(p)
    w = FP.cells_within(r, LON0, LAT0, 180)
    FP.cellmap!(fig[fldmod1(n, 4)[1], fldmod1(n, 4)[2]], r; cells = w, origin = (LON0, LAT0),
                outline = true, title = NAME[m],
                subtitle = @sprintf("%.1f m cells", sqrt(mean(r.mesh.area))),
                axis = (limits = ((-150, 150), (-150, 150)), xlabel = "", ylabel = ""))
end
fig

## 2. Geometry as the solver sees it

`FA.mesh_quality` measures each Carlisle mesh the way the solver sees it
(faces projected about their own latitude, as `A5Grid` does): the valence of
interior cells, how uniform the areas are, the non-orthogonality, and the
face directions. The moments m2, m4, m6 summarise those directions: a square
lattice has m4 = 1 (four face directions), a regular hexagonal one m6 = 1;
a mesh with no preferred directions has all three near 0.

In [ ]:
Q = Dict{String,Any}()
for m in MESHES
    p = carl_path(m, "li"); isfile(p) || continue
    Q[m] = FA.mesh_quality(FA.load_run(p).mesh)
end
rows = map(filter(m -> haskey(Q, m), MESHES)) do m
    q = Q[m]; o = q.orientation                     # m2, m4, m6
    val = join(("$k: $(round(Int, 100 * v / q.n_interior))%" for (k, v) in sort(collect(q.valence)) if v / q.n_interior >= 0.01), ", ")
    (NAME[m], q.n_cells, fr(sqrt(q.area_mean), 1), f(q.area_cv, 4), val,
     fr(q.nonorth_median, 1), fr(q.nonorth_p95, 1), fr(q.nonorth_max, 1), fr(o[1]), fr(o[2]), fr(o[3]))
end
mdtable(["Mesh", "Cells", "Cell size (m)", "Area CV", "Valence (interior)", "Non-orth. median (°)",
         "p95", "max", "m2", "m4", "m6"], rows)

The figure shows, per mesh, the distribution of non-orthogonality over the
faces (left of each pair) and of face directions — the azimuth of each
face normal, mod 180°, weighted by face length (right).

In [ ]:
fig = Figure(size = (1100, 1000))
ms = filter(m -> haskey(Q, m), MESHES)
for (n, m) in enumerate(ms)
    F = Q[m].faces
    row, col = fldmod1(n, 3)
    gl = fig[row, col] = GridLayout()
    a1 = Axis(gl[1, 1]; title = NAME[m], xlabel = "non-orth. (°)", ylabel = "", yticklabelsvisible = false,
              limits = ((0, 45), nothing))
    hist!(a1, F.nonorth_deg; bins = 0:1.5:45, color = mstyle(m).color, normalization = :probability)
    a2 = Axis(gl[1, 2]; xlabel = "face normal (°)", yticklabelsvisible = false, limits = ((0, 180), nothing),
              xticks = 0:45:180)
    hist!(a2, mod.(F.azimuth_deg, 180); bins = 0:3:180, weights = F.width, color = FP.MUTED,
          normalization = :probability)
end
fig

Reading the geometry:

- **Only the planar meshes are orthogonal.** Cartesian, rotated Cartesian,
  hexagons and the Voronoi meshes (orthogonal by construction) have 0°. The
  global grids are not, at 55° N, because of their projections: H3 about 5°,
  rHEALPix 7°, IGEO7 and ISEA3H 10–16°, HEALPix 33° (its polar rhombi), A5
  20–37°. The solver's face-gradient correction (`--face-flux-method
  diamond`) exists for this.
- **Face directions:** the square meshes and rHEALPix have four (m4 ≈ 1),
  the hexagonal meshes six (m6 ≈ 1), HEALPix two skewed pairs (m2 = 0.6).
  A5's are spread over several directions. The Voronoi meshes are seeded
  from a jittered square lattice and keep a memory of it: their face
  normals cluster at 0° and 90° (m4 = 0.56 for the centroidal mesh, whose
  smoothing pulls the cells back towards squares — see the gallery; 0.38
  for the jittered one, the most evenly spread of all).
- **Areas:** the equal-area grids are uniform to about 2e-6; the planar
  meshes vary by 2e-4 (they are equal-area in a local plane, not on the
  sphere); the Voronoi cells by 5–15% by design.

## 3. The cases and their runs

Four registry cases run the benchmarks on every mesh
(`cases/registry/mesh_families.jl`; `docs/TEST_CASES.md` §8). Each makes its
own meshes; the commands of one Carlisle job (each mesh's are alike):

In [ ]:
for name in ("mesh_families_point_spread", "mesh_families_planar_slope", "mesh_families_channel", "mesh_families_carlisle")
    c = FC.case(name)
    @printf("%-28s %-8s %3d runs  %s\n", c.name, c.tier, length(c.jobs), c.title)
end
println()
c = FC.case("mesh_families_carlisle")
for p in filter(p -> p.name in ("hex_raw", "hex", "hex_sgs"), c.prep)
    println("prep ", p.name, ": ", join(FC._subst(p.args, c), " "))
end
for j in filter(j -> startswith(j.name, "hex_"), c.jobs)
    println("run  ", j.name, ": FloodModel.jl ", join(FC.job_argv(c, j), " "))
end

Running them (completed runs are reused; with `RERUN_SGS`, the Carlisle SGS
runs are made again). Each case prints its checks: the physical checks the
case defines, then the comparison with its recorded expected values
(`cases/expected/`), which after a deliberate change to the SGS code will
differ for the SGS runs — that is expected.

In [ ]:
const CASES = ["mesh_families_point_spread", "mesh_families_planar_slope", "mesh_families_channel",
               "mesh_families_carlisle"]
if RERUN_SGS
    FC.run_jobs(PEAK; select = ["a5_sgs"], parallel = PARALLEL, threads = THREADS, fresh = true)
    FC.run_jobs(MFC; select = j -> endswith(j.name, "_sgs"), parallel = PARALLEL, threads = THREADS, fresh = true)
end
if RUN_CASES || RERUN_SGS
    for name in CASES
        FC.run_case(FC.case(name); parallel = PARALLEL, threads = THREADS)
    end
end
M = Dict(name => metrics(name) for name in CASES)
println("metrics read: ", join(sort(collect(keys(M))), ", "))

## 4. Isotropy: a point source on a flat bed

`mesh_families_point_spread`: 10 m³/s for 1 h at the centre of a closed,
flat 1.5 km square at (0°, 0°), on every mesh. Water spreads radially; a
mesh that favours its face directions spreads it further along them. The
metrics, of the depth field after 1 h about the source:

- **A4, A6**: the four- and six-fold harmonics of depth with direction (0
  for a circle);
- **front CV**: the coefficient of variation of the front radius over
  direction sectors;
- **circularity** of the wet area's hull (1 for a circle).

In [ ]:
ps = M["mesh_families_point_spread"]
rows = [(NAME[m], f(ps[m]["cell_size_m"], 1), f(ps[m]["A4"], 4), f(ps[m]["A6"], 4),
         f(ps[m]["cv_radius"], 3), f(ps[m]["circularity"], 3)) for m in MESHES]
mdtable(["Mesh", "Cell size (m)", "A4", "A6", "Front CV", "Circularity"], rows)

In [ ]:
PS = FC.case("mesh_families_point_spread")
fig = Figure(size = (1150, 830))
for (n, m) in enumerate(MESHES)
    r = FA.load_run(FC.job_output(PS, only(filter(j -> j.name == m, PS.jobs))))
    src = FP.run_sources(r)[1]
    h = FA.depth(r, FA.nframes(r))
    FP.cellmap!(fig[fldmod1(n, 4)...], r, [x > 0.005 ? x : NaN for x in h];
                cells = FP.cells_within(r, src.lon, src.lat, 620), origin = (src.lon, src.lat),
                colormap = FP.SEQUENTIAL, colorrange = (0, 0.12), colorbar = false, title = NAME[m],
                axis = (limits = ((-600, 600), (-600, 600)), xlabel = "", ylabel = ""))
end
Colorbar(fig[:, 5]; colormap = cgrad(FP.SEQUENTIAL), limits = (0, 0.12), label = "Depth after 1 h (m)",
         height = Relative(0.5))
fig

Every mesh spreads nearly circularly (circularity > 0.98). What bias there
is follows the face directions: a four-fold term on the meshes with four
face directions (Cartesian, rHEALPix; and HEALPix, whose equatorial cells
are near-squares turned 45°, so A4 < 0), a weaker six-fold term on the
hexagonal ones, and neither on A5. HEALPix's front is the least round.

## 5. Directional bias: a point source on a slope

`mesh_families_planar_slope`: `planar_slope` on every mesh — 0.1 m³/s onto a
plane sloping 0.1% from west to east at 51° N, for 20 h. The plume should
run due east and be symmetric about the source's row:

- **asym**: the north/south volume asymmetry, (N − S)/(N + S), outside a
  narrow band about the source row;
- **deflection**: the plume's direction against east (°);
- **mean flow direction**: the volume-weighted mean velocity direction.

Cartesian and hexagons are mirror-symmetric about the source row, so must be
symmetric (a check of the solver); every other mesh is not.

In [ ]:
pl = M["mesh_families_planar_slope"]
rows = [(NAME[m], f(pl[m]["asym_5h"], 3), f(pl[m]["asym_20h"], 3), f(pl[m]["deflection_20h"], 2),
         f(pl[m]["dir_mean_w"], 1), f(pl[m]["dir_frac30"], 2)) for m in MESHES]
mdtable(["Mesh", "asym 5 h", "asym 20 h", "deflection 20 h (°)", "mean flow dir. (°)", "cells within ±30°"], rows)

In [ ]:
PL = FC.case("mesh_families_planar_slope")
fig = Figure(size = (1150, 720))
for (n, m) in enumerate(MESHES)
    r = FA.load_run(FC.job_output(PL, only(filter(j -> j.name == m, PL.jobs))))
    src = FP.run_sources(r)[1]
    h = FA.depth(r, FA.nframes(r))
    x, y = FA.local_xy(r.mesh, src.lon, src.lat)
    w = [i for i in eachindex(x) if -300 <= x[i] <= 3300 && abs(y[i]) <= 650]
    FP.cellmap!(fig[fldmod1(n, 3)...], r, [v > 0.001 ? v : NaN for v in h]; cells = w, origin = (src.lon, src.lat),
                colormap = FP.SEQUENTIAL, colorrange = (0, 0.02), colorbar = false, title = NAME[m],
                axis = (limits = ((-300, 3200), (-600, 600)), xlabel = "", ylabel = ""))
end
Colorbar(fig[:, 4]; colormap = cgrad(FP.SEQUENTIAL), limits = (0, 0.02), label = "Depth after 20 h (m)",
         height = Relative(0.5))
fig

The Voronoi meshes, irregular but orthogonal, are the least biased after
the two symmetric ones; the global grids sit between 0.04 and 0.13, as A5
does — except HEALPix, whose skewed polar-region cells turn the plume by
about 3° (its plume's northern edge is serrated along the rhombi). The mean
flow is within 2.4° of east on every mesh. (The fraction
of moving cells within ±30° of east, which `planar_slope` checks on A5 and
Cartesian, is not checked here: at 22 m the plume has only 22–160 moving
cells, so the radial flow next to the source decides it.)

## 6. Orientation: a straight channel one cell wide

`mesh_families_channel`: the `channel` series at 22 m (A5 res 18) — a 20 m
wide straight channel, about one cell wide, turned to 0°, 15°, 30° and 45°,
with SWE and local-inertial. Δη is the in-channel stage error against
normal flow (m): positive means the mesh conveys too little and the water
stands too high. A mesh whose result depends on the channel's orientation
is anisotropic at the cell scale; *spread* = max − min of Δη over the four
orientations.

In [ ]:
ch = M["mesh_families_channel"]
deta(m, a, s) = ch["chan_$(m)_r18_a$(lpad(a, 2, '0'))_o00_$s"]["deta"]
const ANGLES = [0, 15, 30, 45]
fig = Figure(size = (1000, 420))
for (k, (s, lab)) in enumerate((("li", "Local-inertial"), ("swe", "SWE")))
    ax = Axis(fig[1, k]; title = lab, xlabel = "Channel orientation (°)", ylabel = k == 1 ? "Δη (m)" : "",
              xticks = ANGLES, limits = (nothing, (-0.2, 1.4)))
    for m in MESHES
        st = mstyle(m)
        scatterlines!(ax, ANGLES, [deta(m, a, s) for a in ANGLES]; color = st.color, linestyle = st.linestyle,
                      marker = st.marker, label = NAME[m])
    end
    k == 2 && Legend(fig[1, 3], ax; framevisible = false)
end
fig

In [ ]:
rows = map(MESHES) do m
    li = [deta(m, a, "li") for a in ANGLES]; sw = [deta(m, a, "swe") for a in ANGLES]
    (NAME[m], f(sqrt(mean(li .^ 2)), 2), f(maximum(li) - minimum(li), 2), f(sqrt(mean(sw .^ 2)), 2),
     f(maximum(sw) - minimum(sw), 2))
end
mdtable(["Mesh", "LI rms Δη (m)", "LI spread (m)", "SWE rms Δη (m)", "SWE spread (m)"], rows)

At one cell per channel width every mesh loses conveyance, and each loses
least where its faces line up with the channel: the rotated grid at 30°;
Cartesian, rHEALPix and the hexagons at 0°. Hence the regular meshes'
large spreads (0.5–1 m). A5 and HEALPix hardly depend on orientation, but
sit at about the Cartesian mean; H3 does best overall, helped by its 25%
smaller cells. SWE agrees with local-inertial throughout, so this is the
meshes, not the local-inertial equations.

## 7. A real flood: Carlisle against the 5 m references

`mesh_families_carlisle`: the Carlisle flood of January 2005 from its
hydrographs to the Eden peak (55.5 h), with local-inertial and SGS on every
mesh (A5 from `carlisle_peak`). Each run is compared, on its own cells, with
the 5 m runs of `carlisle_peak_fine` aggregated onto them: the 5 m SWE run
(the benchmark) and the 5 m local-inertial run (what SGS, local-inertial on
the 5 m terrain, approximates; `docs/HYDRAULICS.md` §15.14). At the peak,
55.5 h:

- **storage**: the run's stored volume over the benchmark's;
- **level rms**: the rms water-level difference over cells wet in either;
- **CSI**: the critical success index of the wet extent (depth > 0.1 m).

In [ ]:
mc = M["mesh_families_carlisle"]
ckey(m, s) = m == "a5" ? "peak:a5_$s" : "$(m)_$s"
cm(m, s, k) = (d = get(mc, ckey(m, s), nothing); d === nothing ? NaN : Float64(get(d, k, NaN)))
rows = [(NAME[m], f(cm(m, "li", "cell_size_m"), 1),
         f(cm(m, "li", "swe5_storage_ratio_w1"), 3), f(cm(m, "li", "swe5_level_rms_w1"), 3), f(cm(m, "li", "swe5_csi_w1"), 3),
         f(cm(m, "sgs", "swe5_storage_ratio_w1"), 3), f(cm(m, "sgs", "swe5_level_rms_w1"), 3), f(cm(m, "sgs", "swe5_csi_w1"), 3),
         f(cm(m, "sgs", "li5_level_rms_w1"), 3)) for m in MESHES]
mdtable(["Mesh", "Cell size (m)", "LI storage", "LI level rms (m)", "LI CSI",
         "SGS storage", "SGS level rms (m)", "SGS CSI", "SGS rms vs 5 m LI (m)"], rows)

In [ ]:
fig = Figure(size = (1100, 430))
ys = 1:length(MESHES)
for (k, (key, lab)) in enumerate((("swe5_storage_ratio_w1", "Storage / 5 m SWE"),
                                  ("swe5_level_rms_w1", "Level rms vs 5 m SWE (m)"),
                                  ("swe5_csi_w1", "CSI vs 5 m SWE")))
    ax = Axis(fig[1, k]; xlabel = lab, yticks = (ys, [NAME[m] for m in MESHES]), yreversed = true,
              yticklabelsvisible = k == 1)
    scatter!(ax, [cm(m, "li", key) for m in MESHES], ys; color = FP.SERIES[1], marker = :circle, label = "LI")
    scatter!(ax, [cm(m, "sgs", key) for m in MESHES], ys; color = FP.SERIES[2], marker = :utriangle, label = "SGS")
    k == 3 && Legend(fig[2, 1:3], ax; orientation = :horizontal, framevisible = false)
end
fig

**Local-inertial:** the mesh family matters little at matched cell size on
a real flood — on every ~22 m mesh 0.92–0.97 of the benchmark's storage,
0.25–0.30 m level rms and CSI 0.82–0.85; the finer H3 and rHEALPix meshes
have the best extents. **SGS** stores 0.70–0.79 of the benchmark on every
mesh (HYDRAULICS §15.14: it follows the 5 m local-inertial run, within
0.12–0.27 m rms, and that run itself stores less than SWE). Until 0.9.0 the
checkerboard of §8 made several meshes' numbers noise: the Cartesian and
rotated grids' SGS storage was then 1.06 and 0.96 of the benchmark.

## 8. The SGS checkerboard — a re-check for SGS work

Until 0.9.0, on most meshes — the Cartesian grid among them — SGS developed
a cell-scale checkerboard in the water level at Carlisle: neighbouring cells
alternated high and low (local-inertial on the same meshes is smooth). It
was found by `mesh_families_carlisle` in 0.8.1 and recorded there as an
expected failure. Its cause was the order of a step: the sources were added
before the face fluxes were computed, so the Eden's inflow cell, lifted
≈ 2.4 m a step, and the adaptive Δt drove each other (HYDRAULICS §15.15).
Since 0.9.0 SGS adds them after the flux update (`--source-order post`).
This section measures the checkerboard per mesh; re-run it (with
`RERUN_SGS = true`, §0) after a change to the SGS code.

**What is measured.** For each fully wet interior cell (it and all its
neighbours deeper than 5 cm), the residual rᵢ = ηᵢ − mean(η of its
neighbours): small and smooth for a smooth surface, ±(amplitude) for a
checkerboard (`FA.wse_residual`, `FA.noise_metrics`). Per run:

- **extrema**: the fraction of those cells that are strict local extrema of
  the water level with |rᵢ| > 2 mm — about 0 for a smooth surface,
  approaching 1 for a perfect checkerboard. The case's check: below
  `FC.CHECKERBOARD_EXTREMA` (0.05) at the peak;
- **r rms**: the rms residual (mm);
- **storage drops**: the sum of the falls of the stored volume between
  output frames on the rising limb (40 h to the peak), over the stored
  volume at the peak — 0 while the inflow rises, unless the water
  oscillates. The check: 0;
- **max depth**: the largest depth in the peak window (49.5–55.5 h), which
  noise inflates.

### 8.1 Which code made the runs

The case runner does not know about code changes. Check here that the SGS
runs were made by the code you mean to test.

In [ ]:
head = try strip(read(`git -C $REPO rev-parse --short HEAD`, String)) catch; "?" end
println("this checkout: version ", strip(read(joinpath(REPO, "VERSION"), String)), ", commit ", head)
rows = map(MESHES) do m
    p = carl_path(m, "sgs")
    isfile(p) || return (NAME[m], "missing", "", "", "")
    ri = FA.FO.run_info(p)
    (NAME[m], get(ri, "flooda5_version", "?"), first(string(get(ri, "git_commit", "?")), 7) *
     (get(ri, "git_dirty", false) == true ? " (+ local changes)" : ""),
     first(string(get(ri, "created_utc", "?")), 16), string(get(get(ri, "summary", Dict()), "finished_utc", "?")))
end
mdtable(["SGS run", "FloodA5 version", "Commit", "Started (UTC)", "Finished (UTC)"], rows)

### 8.2 Per mesh

The checkerboard measures for every mesh at the peak, SGS and
local-inertial side by side; *smooth* = no storage drop on the rising limb
and extrema below the bound.

In [ ]:
const EXT = FC.CHECKERBOARD_EXTREMA
smooth(m, s) = cm(m, s, "storage_drop_rel") == 0 && cm(m, s, "noise_extrema_w1") < EXT
rows = map(MESHES) do m
    (NAME[m], f(cm(m, "sgs", "noise_extrema_w1"), 3), f(cm(m, "sgs", "noise_r_rms_w1_mm"), 0),
     f(cm(m, "sgs", "storage_drop_rel"), 3), f(cm(m, "sgs", "max_depth_peak_m"), 1),
     smooth(m, "sgs") ? "smooth" : "CHECKERBOARD",
     f(cm(m, "li", "noise_extrema_w1"), 3), f(cm(m, "li", "noise_r_rms_w1_mm"), 0), f(cm(m, "li", "max_depth_peak_m"), 1),
     smooth(m, "li") ? "smooth" : "CHECKERBOARD")
end
display(mdtable(["Mesh", "SGS extrema", "SGS r rms (mm)", "SGS storage drops", "SGS max depth (m)", "SGS",
                 "LI extrema", "LI r rms (mm)", "LI max depth (m)", "LI"], rows))
good = [NAME[m] for m in MESHES if smooth(m, "sgs")]
bad = [NAME[m] for m in MESHES if !smooth(m, "sgs")]
println("SGS smooth on ", length(good), " of ", length(MESHES), " meshes: ", join(good, ", "))
isempty(bad) || println("SGS checkerboard on: ", join(bad, ", "))
println("LI smooth on ", count(m -> smooth(m, "li"), MESHES), " of ", length(MESHES), " meshes")

### 8.3 Through the event

The extrema fraction every 2 hours, SGS (coloured) and local-inertial (grey).
Before 0.9.0 the checkerboard appeared within hours on some meshes
(rHEALPix 1.5 h, rotated Cartesian 3 h) and only near the peak on others
(hexagons 50 h), always starting next to an inflow cell.

In [ ]:
fig = Figure(size = (1000, 430))
ax = Axis(fig[1, 1]; xlabel = "Time (h)", ylabel = "Fraction of wet cells that are local extrema",
          limits = ((0, 56), (0, nothing)))
hlines!(ax, [EXT]; color = FP.MUTED, linestyle = :dot)
for m in MESHES, s in ("li", "sgs")
    p = carl_path(m, s); isfile(p) || continue
    se = FA.noise_series(FA.load_run(p); every = 4)
    t = [e.t / 3600 for e in se]; e = [e.m.extrema for e in se]
    st = mstyle(m)
    s == "li" ? lines!(ax, t, e; color = (FP.MUTED, 0.5), linewidth = 1) :
                lines!(ax, t, e; color = st.color, linestyle = st.linestyle, label = NAME[m])
end
Legend(fig[1, 2], ax; framevisible = false)
fig

### 8.4 What it looks like

The residual rᵢ at the peak over 1 km of central Carlisle, where the
checkerboard was strongest, SGS above and local-inertial below, on one
colour scale (cells not fully wet are grey). A checkerboard shows as
alternating red and blue cells; a smooth surface as pale cells with smooth
patterns where the water slopes.

In [ ]:
const T_PEAK = FC.CARLISLE_PEAK_WINDOW[2]
function residual_maps(s)
    fig = Figure(size = (1150, 820))
    for (n, m) in enumerate(MESHES)
        p = carl_path(m, s); isfile(p) || continue
        r = FA.load_run(p)
        res = FA.wse_residual(r, FA.frame_at(r, T_PEAK; tol = 300))
        FP.cellmap!(fig[fldmod1(n, 4)...], r, res.r; cells = FP.cells_within(r, LON0, LAT0, 520),
                    origin = (LON0, LAT0), colorrange = (-0.25, 0.25), colorbar = false,
                    title = NAME[m] * (s == "sgs" ? " · SGS" : " · LI"),
                    subtitle = @sprintf("extrema %.3f (whole domain)", count(res.extremum) / count(isfinite, res.r)),
                    axis = (limits = ((-500, 500), (-500, 500)), xlabel = "", ylabel = ""))
    end
    Colorbar(fig[:, 5]; colormap = cgrad(FP.DIVERGING), limits = (-0.25, 0.25), highclip = FP.DIVERGING[end],
             lowclip = FP.DIVERGING[1], label = "η − mean of neighbours (m)", height = Relative(0.5))
    return fig
end
residual_maps("sgs")

In [ ]:
residual_maps("li")

### 8.5 Storage on the rising limb

Stored volume from 40 h to the peak, SGS (coloured) and local-inertial
(grey). The inflow rises throughout, so the stored volume should too. In
0.8.x SGS oscillated on rHEALPix and the rotated grid, and on the Cartesian
grid it climbed far above every other mesh once the checkerboard set in
(about 46 h); since 0.9.0 it rises smoothly on every mesh.

In [ ]:
fig = Figure(size = (1000, 430))
ax = Axis(fig[1, 1]; xlabel = "Time (h)", ylabel = "Stored volume (10⁶ m³)")
for m in MESHES, s in ("li", "sgs")
    p = carl_path(m, s); isfile(p) || continue
    r = FA.load_run(p)
    ks = [k for k in 1:FA.nframes(r) if r.t[k] >= FC.CARLISLE_RISE - 300]
    st = mstyle(m)
    v = [FA.stored_volume(r, k) / 1e6 for k in ks]
    s == "li" ? lines!(ax, r.t[ks] ./ 3600, v; color = (FP.MUTED, 0.5), linewidth = 1) :
                lines!(ax, r.t[ks] ./ 3600, v; color = st.color, linestyle = st.linestyle, label = NAME[m])
end
Legend(fig[1, 2], ax; framevisible = false)
fig

**Since 0.9.0** SGS is smooth on every mesh: at the peak at most 1.6% of
the wet cells are local extrema (rotated Cartesian; local-inertial ≤ 0.8%),
the stored volume rises throughout, and the maximum depth is 6.6–6.8 m.
**In 0.8.x** it was smooth only on A5, ISEA3H and the jittered Voronoi mesh;
it checkerboarded over 8–23% of the wet cells on hexagons, centroidal
Voronoi, H3, IGEO7 and HEALPix, and over 36–42% on rHEALPix and the rotated
and plain Cartesian grids, where the stored volume also oscillated or ran
away on the rising limb and the maximum depth jumped to 11 m. It started next
to the inflow cells on every mesh: how early depended on how much the inflow
lifted its cell each step and on how strongly the time step followed it, not
on regularity or orthogonality (HYDRAULICS §15.15). After a change to SGS,
the tables above say whether every mesh is still smooth.

## 9. Cost

From the Carlisle runs: cells, faces, time steps and wall time. The wall
times are indicative only: the runs shared one machine, two or three at a
time, and A5's were made separately (`carlisle_peak`). Roughly, the cost of a
step follows the faces per cell — squares (2) cheapest, hexagons and
Voronoi cells (3) about 1.5–2 times dearer — and the number of steps
follows the smallest cells (H3, rHEALPix) and, for SGS, the deepest water:
SGS takes 1.5–2.3 times as many steps (Courant 0.5 against 0.7, and the
time step of cells whose faces carry a channel; up to 2.7 times in 0.8.x,
where it checkerboarded).

In [ ]:
rows = []
for m in MESHES, s in ("li", "sgs")
    p = carl_path(m, s); isfile(p) || continue
    ri = FA.FO.run_info(p); su = get(ri, "summary", Dict())
    nc = haskey(Q, m) ? Q[m].n_cells : missing
    nfc = haskey(Q, m) ? Q[m].n_faces : missing
    steps = get(su, "n_steps", NaN); wall = get(su, "wall_simulation_s", NaN)
    push!(rows, (NAME[m], uppercase(s), nc, ismissing(nfc) ? "" : fr(nfc / nc, 2), steps,
                 fr(FC.CARLISLE_PEAK_WINDOW[2] / steps, 2), get(ri, "threads", "?"), round(Int, wall / 60),
                 fr(1e6 * wall * get(ri, "threads", 1) / (steps * nc), 2)))
end
mdtable(["Mesh", "Model", "Cells", "Faces per cell", "Steps", "Mean Δt (s)", "Threads", "Wall (min)",
         "thread-µs per cell-step"], rows)

## 10. Advantages and disadvantages

From the evidence above (and `docs/MESHES.md`):

| Mesh | For | Against |
|---|---|---|
| Cartesian | Raster-compatible: DEMs, other models (LISFLOOD-FP), GIS; orthogonal; fewest faces per cell; SGS closest to the 5 m local-inertial run (0.13 m rms) | Four face directions: the strongest four-fold bias and orientation dependence; not equal-area as a lon/lat grid |
| rotated Cartesian | Shows that a square grid's bias turns with it | As Cartesian, turned |
| hexagons (planar) | Orthogonal; six directions: weaker bias than squares | Six directions still show in a narrow channel; 3 faces per cell cost more; local to a plane |
| Voronoi | Orthogonal, irregular: least directional bias on the slope; can follow features | Irregular areas; must be generated per site; Lloyd smoothing pulls it back towards its seed lattice |
| A5 | Equal-area global grid; spread face directions: nearly orientation-independent in the channel | Most non-orthogonal (20–37°); children do not tile the parent; coarse ×4 levels |
| H3 | The largest ecosystem; nearly orthogonal (5°) | Not equal-area; ×7 levels (here 25% finer than the target); children do not tile the parent |
| IGEO7 | Equal-area hexagons | ×7 levels; 10–16° non-orthogonal |
| ISEA3H | Equal-area hexagons; the finest level steps (×3) | 13.5° non-orthogonal; children do not tile the parent |
| HEALPix | Equal-area; children tile the parent; nearly orientation-independent in the channel | Polar rhombi above 41.8°: 33° non-orthogonal, the largest bias on the slope and the least round spread |
| rHEALPix | Equal-area; children tile the parent; nearly orthogonal (7°) | Squares again (four-fold bias, orientation dependence); ×9 levels |

## 11. Conclusions

- The solver runs on all eleven meshes unchanged, with mass conserved to
  round-off, and the local-inertial results at a matched cell size are close
  on a real flood: the choice of mesh family is secondary to cell size there.
- In idealised flows each mesh shows the directions of its faces. Meshes
  with many or spread face directions (A5, Voronoi, HEALPix in the channel)
  depend least on flow direction; square grids most.
- No global grid is orthogonal at British latitudes; HEALPix is the most
  distorted there (33°), and it shows.
- SGS had a mesh-dependent checkerboard (§8) until 0.9.0, smooth only on A5,
  ISEA3H and jittered Voronoi: the sources entered before the face fluxes,
  and the inflow cell and the adaptive time step drove each other. It is
  smooth on every mesh now, and stores 0.70–0.79 of the 5 m SWE benchmark on
  all of them: the gap is the local-inertial hydraulics (HYDRAULICS §15.14),
  the next SGS work. Re-run §8 after it.